# 02 — LLM-as-Judge Baseline

**Goal:** Establish a rigorous Claude Sonnet 4.6 baseline on the Banking77 test set (3,080 queries). Versioned prompts, full evaluation, calibration analysis.

## Cost-controlled iteration strategy

Running Sonnet 4.6 on the full 3,080-row test set with ~1,500 tokens of label-list prompt + the query costs roughly **$15 per eval run**. Prompt iteration on the full set burns budget fast.

Instead:
1. Iterate the prompt on the **300-row dev slice** (carved from the train pool in notebook 01). Roughly **$1.50 per iteration**.
2. Once the prompt converges to `v_final`, run **one** full eval on the 3,080-row test set (~$15).

Total budget: ~$25 across 5–8 prompt iterations.

## Sections

1. Load test set, label list, dev slice
2. Versioned prompt loader (prompts live in `src/prompts/`)
3. Iterate prompt on the 300-row dev slice (v1 → v2 → … → v_final)
4. Final eval on the full 3,080-row test set (one run, on v_final only)
5. Parse outputs, handle malformed responses (intent name not in label list)
6. Evaluate: accuracy, macro-F1, per-intent F1, confusion matrix, top confusion pairs
7. Latency + cost measurement
8. Save predictions to `results/llm_baseline_predictions.parquet`

## 1. Load test set, label list, dev slice

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path('..').resolve()))

from src.data import load_test_set, load_dev_slice, load_label_names

# TODO: test_df = load_test_set(); dev_df = load_dev_slice(); label_names = load_label_names()

## 2. Versioned prompt loader

Every prompt version lives as a text file in `src/prompts/` so changes are diff-able in git. `v1.txt` ships with the scaffold; create `v2.txt`, `v3.txt`, etc. as you iterate.

**Design note:** with 77 intents in the prompt, the input context per call is ~1,500 tokens of label list + the query. That dominates the cost. Possible v2+ optimisations: hierarchical routing (broad category → fine intent), label-set caching via Anthropic's prompt caching, or batch classification.

In [ ]:
from src.prompts import load_prompt, list_versions

print('Available prompt versions:', list_versions())

# TODO: PROMPT = load_prompt('v1')
# TODO: format the label_list block once; PROMPT.format(label_list=..., query=...) per call

## 3. Iterate prompt on the dev slice

For each prompt version v1, v2, …, run Claude on the 300-row dev slice and compute macro-F1. Compare versions. Iterate until additional prompt changes stop helping.

**Important:** never look at the test set during prompt iteration. The dev slice is your only signal.

In [ ]:
from anthropic import Anthropic
import time

client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-sonnet-4-6'

# TODO: function run_eval(prompt_template, df) -> predictions_df
#         loops over df rows with retries + rate-limit handling, saves partial results every 50 queries
# TODO: for each prompt version: predictions = run_eval(load_prompt(version), dev_df)
#         macro_f1 = ... ; print(version, macro_f1)
# Pick the best-performing prompt as v_final

## 4. Final eval on the full test set

Run v_final once on all 3,080 test rows. This is the **only** time the test set is touched during prompt work. ~10–15 min wall-clock at API rate limits, ~$15 spend.

In [ ]:
# TODO: FINAL_PROMPT_VERSION = 'v_final'  # or whichever version won on the dev slice
# TODO: test_predictions = run_eval(load_prompt(FINAL_PROMPT_VERSION), test_df)

## 5. Parse outputs

In [ ]:
# TODO: parse JSON, flag malformed responses + intent-name hallucinations (Claude invents an intent not in the 77-class list)
# Handling: retry once with a stricter prompt, then drop unparseable rows from the metric (but report the drop rate)

## 6. Evaluate

In [ ]:
from src.eval import compute_metrics
import matplotlib.pyplot as plt
import seaborn as sns

# TODO: metrics = compute_metrics(test_df['label'], test_predictions['pred_label'])
# TODO: confusion matrix (77x77 — visualise top-20 most-confused pairs)
# TODO: identify systematic confusions (e.g. card_arrival vs card_delivery_estimate)

## 7. Latency + cost

In [ ]:
# Posted Sonnet 4.6 pricing at time of writing (verify before publishing):
#   Input: $3.00 / MTok ; Output: $15.00 / MTok
# TODO:
# - Record latency per call (median + p95)
# - Sum input + output tokens from response usage; compute cost
# - Document: cost per 1k inferences, cost per query, total experiment cost (dev iterations + final run)

## 8. Save predictions

Saved to `results/llm_baseline_predictions.parquet` so notebook 04 can join against DistilBERT predictions for the bootstrap test and failure-mode analysis.

In [ ]:
# TODO: save with columns [text, true_label, pred_label, pred_intent_name, raw_response, latency_ms, input_tokens, output_tokens]